# 01 – Data Preprocessing

Loads the nationwide H3 grid dataset (parquet) produced by DuckDB,
translates Japanese park names in the `NAME` column to English slugs,
applies one-hot encoding to categorical geological and landform variables,
and saves the processed dataset for use in downstream modeling.

After this step all Japanese text is removed from the dataset;
subsequent notebooks work exclusively with English identifiers.

**Input**  : `data/interim/h3_jpn_res9_source_imputed.parquet`  
**Output** : `data/interim/h3_jpn_res9_processed.parquet`

Corresponds to *Section 2.2 – Data preprocessing* in the manuscript.

## Imports and configuration

In [1]:
import pandas as pd

from config import DATA_DIR, PARK_NAME_MAP

## Load nationwide H3 dataset

In [2]:
input_path = DATA_DIR / "h3_jpn_res9_source_imputed.parquet"
df_all = pd.read_parquet(input_path)

print(f"Loaded: {df_all.shape[0]:,} rows x {df_all.shape[1]} columns")
df_all.head()

Loaded: 4,051,335 rows x 14 columns


,h3_9,NAME,ZONE,np_class,bichikei_en,group_en,lithology1,h3int,elev_mean,slopemean,chishitsu_age,prec_year,ave_temp_y,max_snow_y
0,892e76a4173ffff,NaN,NaN,0,mountains,Igneous rocks,massive granite island arc & continental,617810542453850111,580.327381,24.880113,83600000.0,28994.0,92.0,251.0
1,892e290d3cfffff,NaN,NaN,0,valley bottom lowland,Sedimentary rocks,"valley floor, intermountain basin, river & co...",617805210559971327,313.250000,2.524655,11700.0,13152.0,112.0,16.0
2,892e6399e53ffff,NaN,NaN,0,back marsh,Sedimentary rocks,natural levee deposits,617809234047008767,90.143791,3.352369,11700.0,17407.0,138.0,37.0
3,892ee611a67ffff,NaN,NaN,0,mountains,Igneous rocks,andesite & basaltic andesite lava & pytoclasti...,617818199725441023,425.338308,26.281315,28100000.0,16962.0,61.0,436.0
4,892e6d8c2bbffff,NaN,NaN,0,mountains,Metamorphic rocks,pelitic schist chlorite zone of high P/T regio...,617809917557604351,720.430657,23.841592,113000000.0,20839.0,122.0,0.0


## Translate park names to English slugs

The source GIS data uses Japanese park names in the `NAME` column.
These are translated to English slugs here so that all downstream
notebooks and output files are free of Japanese text.

In [3]:
df_all["NAME"] = df_all["NAME"].map(PARK_NAME_MAP)

# Sanity check: rows outside any national park have NAME == NaN — expected.
# Rows inside a park that failed to map indicate a name mismatch.
park_rows = df_all[df_all["np_class"] == 1]
unmapped  = park_rows[park_rows["NAME"].isna()]
if not unmapped.empty:
    raise ValueError(f"{len(unmapped)} park rows could not be mapped to English slugs.")

print(f"Park name translation complete.")
print(f"Unique slugs ({df_all['NAME'].nunique()}): {sorted(df_all['NAME'].dropna().unique())}")

Park name translation complete.
Unique slugs (35): ['akan', 'amami', 'ashizuri', 'aso', 'bandai', 'chichibu', 'chubusangaku', 'daisen', 'fuji', 'hakusan', 'hidaka', 'iriomote', 'ise', 'jyoshinetsu', 'kerama', 'kirishima', 'kushiro', 'minamialps', 'myoko', 'nikko', 'ogasawara', 'oze', 'rishiri', 'saikai', 'sanin', 'sanriku', 'setonaikai', 'shikotsu', 'shiretoko', 'taisetsu', 'towada', 'unzen', 'yakushima', 'yambaru', 'yoshino']


## One-hot encoding of categorical variables

Landform classification (`bichikei_en`) and geological group (`group_en`)
are one-hot encoded. The first category is dropped to avoid multicollinearity.

See *Section 2.2.1 – Data integration* in the manuscript.

In [4]:
categorical_cols = ["bichikei_en", "group_en"]

df_all = pd.get_dummies(df_all, columns=categorical_cols, drop_first=True)

print(f"After encoding: {df_all.shape[0]:,} rows x {df_all.shape[1]} columns")

After encoding: 4,051,335 rows x 42 columns


## Save processed dataset

In [5]:
output_path = DATA_DIR / "h3_jpn_res9_processed.parquet"
df_all.to_parquet(output_path, index=False)

print(f"Saved: {output_path}")

Saved: ..\data\interim\h3_jpn_res9_processed.parquet
